# cMCF batch testing: arbitrary genus-0 mesh -> SHP artifacts

Runs the locked-in cMCF pipeline (repair + curvature remesh -> cMCF -> Mobius
centre -> Adam area-equalize -> stretch-aligned anisotropic refinement -> SHP)
over a folder of test meshes and, for each, writes **6 artifacts** (incl. the parametric sphere `_f_param_sphere.off`, all `.off` with clean outward normals):

- **(a)** optimized starting mesh -> `<name>_a_optimized.off`
- **(b)** SHP coefficients -> `<name>_b.shp3`
- **(c)** rotation+scale **invariant** SHP coefficients -> `<name>_c_invariant.shp3`
- **(d)** mesh reconstructed from (b) -> `<name>_d_recon.off`
- **(e)** mesh reconstructed from (c) -> `<name>_e_recon_invariant.off`

Plus a robust **per-degree energy descriptor** (the true rotation/scale
invariant) for shape distance, and a summary table (bijective? folds? RMS?).

Notes
- Each mesh runs in a try/except so one bad mesh doesn't stop the batch.
- **(c) caveat:** the invariant *pose* (first-order-ellipsoid alignment) is
  exact for asymmetric shapes but ambiguous up to symmetry for near-symmetric
  ones (e.g. the mushroom's axisymmetric cap). For shape *distance*, use the
  per-degree energy descriptor (robust regardless of symmetry).

## Imports

In [1]:
import numpy as np
import sys, os, glob, importlib, traceback

code_dir = 'C:/Users/Khaled Khairy/Dropbox/Projects/hot/Project_spherical_parameterization/code'
if code_dir not in sys.path:
    sys.path.insert(0, code_dir)

from pySHP.surface_mesh import surface_mesh
from pySHP.shp_surface import shp_surface
from pySHP.utils import readoff, writeoff, kk_sph2cart

import pySHP.level2.cmcf_spherical_parameterization as cmcf
importlib.reload(cmcf)
print('cMCF module:', cmcf.__file__)

cMCF module: C:\Users/Khaled Khairy/Dropbox/Projects/hot/Project_spherical_parameterization/code\pySHP\level2\cmcf_spherical_parameterization.py


## Settings

In [2]:
TEST_DIR = os.path.join(code_dir, 'Matlab', 'shp_toolbox-main', 'shp_toolbox-main',
                        'test_data', 'off', 'test_set')
# Shared parameterization-output folder (in the project root, outside the repo).
OUT_DIR  = os.path.join(os.path.dirname(code_dir), 'Projection_output')
os.makedirs(OUT_DIR, exist_ok=True)

# Which meshes: all .off in TEST_DIR (optionally cap for a quick run).
MESH_PATHS = sorted(glob.glob(os.path.join(TEST_DIR, '*.off')))
MAX_MESHES = None          # e.g. 3 for a quick smoke run; None = all
if MAX_MESHES:
    MESH_PATHS = MESH_PATHS[:MAX_MESHES]

# Pipeline parameters (the locked-in working version).
TARGET_VERTS  = 7000
CURV_STRENGTH = 2.0
CMCF_ITER     = 120
AREA_NITER    = 1200
LAMBDA_SHAPE  = 0.3
ANISO_ROUNDS  = 3
ANISO_SPLIT   = 1.7
L_MAX         = 60         # SHP bandwidth
OVERSAMPLE    = 40         # SHP fit upsampling factor
NICO          = 4          # icosphere subdivisions for reconstructed meshes
VISUALIZE     = True       # show per-mesh 3-D panels
print(f'{len(MESH_PATHS)} meshes; outputs -> {OUT_DIR}')

9 meshes; outputs -> C:/Users/Khaled Khairy/Dropbox/Projects/hot/Project_spherical_parameterization/code\pySHP\tests\cmcf_batch_outputs


## Plotly helpers (same style as the cMCF notebook)

In [3]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def _sphere_xyz(m):
    u, v, w = kk_sph2cart(m.t, m.p, np.ones(len(m.t)))
    return np.column_stack([u, v, w])

SHADE = 0.18
_LIGHT = dict(ambient=1.0 - SHADE, diffuse=SHADE, specular=0.0, roughness=1.0, fresnel=0.0)
_LIGHTPOS = dict(x=1.2e4, y=1.2e4, z=1.2e4)

def _curv_face(m):
    try:
        if getattr(m, 'H', None) is None:
            m.props()
        H = np.abs(np.asarray(m.H, float)); H = np.where(np.isfinite(H), H, 0.0)
        return H[np.asarray(m.F, int)].mean(axis=1)
    except Exception:
        return None

def _mesh_trace(X, F, fs=None, colorscale='Inferno', color='lightblue', showscale=False):
    X = np.asarray(X, float); F = np.asarray(F, int)
    kw = dict(x=X[:, 0], y=X[:, 1], z=X[:, 2], i=F[:, 0], j=F[:, 1], k=F[:, 2],
              flatshading=True, showscale=showscale, lighting=_LIGHT, lightposition=_LIGHTPOS)
    if fs is not None:
        v = np.asarray(fs, float); kw.update(intensity=v, intensitymode='cell', colorscale=colorscale)
        fin = v[np.isfinite(v)]
        if fin.size:
            lo, hi = np.nanpercentile(fin, (2, 98))
            if hi > lo: kw.update(cmin=float(lo), cmax=float(hi))
    else:
        kw.update(color=color)
    return go.Mesh3d(**kw)

def _blank(): return dict(xaxis=dict(visible=False), yaxis=dict(visible=False),
                          zaxis=dict(visible=False), aspectmode='data')

def _shear_face(m):
    sh, _ = surface_mesh.compute_shear_spherical(m.t, m.p, m.F); return sh

def show_result(name, m_param, shp_b, shp_c, nico=4):
    sph = _sphere_xyz(m_param)
    smb, Xb, Fb, *_ = shp_b.get_mesh(nico=nico)
    smc, Xc, Fc, *_ = shp_c.get_mesh(nico=nico)
    fig = make_subplots(rows=1, cols=4, specs=[[{'type': 'scene'}] * 4],
                        subplot_titles=(f'{name}: object (curv)', 'sphere (shear)',
                                        '(d) SHP recon', '(e) invariant recon'))
    fig.add_trace(_mesh_trace(m_param.X, m_param.F, _curv_face(m_param)), 1, 1)
    fig.add_trace(_mesh_trace(sph, m_param.F, _shear_face(m_param)), 1, 2)
    fig.add_trace(_mesh_trace(Xb, Fb, _curv_face(smb)), 1, 3)
    fig.add_trace(_mesh_trace(Xc, Fc, _curv_face(smc)), 1, 4)
    fig.update_layout(height=420, margin=dict(l=0, r=0, t=30, b=0), template='plotly_white',
                      scene=_blank(), scene2=_blank(), scene3=_blank(), scene4=_blank())
    fig.show()
print('helpers ready')

helpers ready


## Per-mesh pipeline + export

In [4]:
def run_one(path, visualize=True):
    name = os.path.splitext(os.path.basename(path))[0]
    rec = {'name': name}
    try:
        X, F = readoff(path)
        m_in = surface_mesh(X, F)
        info = m_in.info(verbose=False)
        rec.update(n_in=len(m_in.X), genus=info.get('genus'),
                   closed=info.get('is_closed'), comps=info.get('n_components'))

        res = cmcf.parameterize_to_sphere(
            m_in, target_verts=TARGET_VERTS, curvature_strength=CURV_STRENGTH,
            cmcf_iter=CMCF_ITER, area_blend=1.0, area_n_iter=AREA_NITER,
            lambda_shape=LAMBDA_SHAPE, aniso_rounds=ANISO_ROUNDS,
            aniso_split_factor=ANISO_SPLIT, fit_shp_L_max=L_MAX,
            shp_oversample=OVERSAMPLE, verbose=False)
        m_param, shp_b = res['mesh'], res['shp']
        d = res['stage2_diag']

        shp_c, cinfo = cmcf.canonicalize_shp(m_param, L_max=L_MAX,
                                             oversample=OVERSAMPLE, verbose=False)

        out = os.path.join(OUT_DIR, name); os.makedirs(out, exist_ok=True)
        cmcf.export_off(os.path.join(out, f'{name}_a_optimized.off'), m_param.X, m_param.F)
        cmcf.export_off(os.path.join(out, f'{name}_f_param_sphere.off'), _sphere_xyz(m_param), m_param.F)  # (f) parametric sphere (same connectivity)
        shp_b.export_shp3(os.path.join(out, f'{name}_b.shp3'))
        shp_c.export_shp3(os.path.join(out, f'{name}_c_invariant.shp3'))
        _, Xb, Fb, *_ = shp_b.get_mesh(nico=NICO)
        cmcf.export_off(os.path.join(out, f'{name}_d_recon.off'), Xb, Fb)
        _, Xc, Fc, *_ = shp_c.get_mesh(nico=NICO)
        cmcf.export_off(os.path.join(out, f'{name}_e_recon_invariant.off'), Xc, Fc)

        E = cmcf.shp_degree_energy(shp_b)     # rotation/scale-invariant descriptor
        rec.update(status='ok', quality=res.get('quality'),
                   kappa=round(res.get('complexity', {}).get('kappa', float('nan')), 2),
                   n_param=len(m_param.X), bijective=d['bijective'],
                   folds=res.get('n_foldovers', d['n_foldovers']),
                   min_q=round(d['min_quality'], 3),
                   max_shear=round(d['max_shear'], 2), area_cov=round(d['area_cov'], 2),
                   shp_rms_pct=round(100 * res.get('shp_rms_rel', float('nan')), 2),
                   semi_axes=[round(x, 3) for x in cinfo['semi_axes']],
                   descriptor=[float(x) for x in E])
        if visualize:
            print(f"  {name}: bijective={d['bijective']} folds={d['n_foldovers']} "
                  f"RMS={rec['shp_rms_pct']}%  -> 6 files in {out}")
            show_result(name, m_param, shp_b, shp_c, nico=NICO)
    except Exception as exc:
        rec['status'] = 'FAIL: ' + str(exc)
        traceback.print_exc()
    return rec
print('run_one ready')

run_one ready


## Run the batch

In [ ]:
results = []
for i, p in enumerate(MESH_PATHS):
    print(f'\n[{i+1}/{len(MESH_PATHS)}] {os.path.basename(p)}')
    results.append(run_one(p, visualize=VISUALIZE))
print('\nbatch done')


[1/9] 1dpx.off
Exported spherical harmonics surface to: C:/Users/Khaled Khairy/Dropbox/Projects/hot/Project_spherical_parameterization/code\pySHP\tests\cmcf_batch_outputs\1dpx\1dpx_b.shp3
  L_max: 16
  Number of coefficients: 289
  Components: 3 (x, y, z)
Exported spherical harmonics surface to: C:/Users/Khaled Khairy/Dropbox/Projects/hot/Project_spherical_parameterization/code\pySHP\tests\cmcf_batch_outputs\1dpx\1dpx_c_invariant.shp3
  L_max: 16
  Number of coefficients: 289
  Components: 3 (x, y, z)
  1dpx: bijective=False folds=3 RMS=0.89%  -> 5 files in C:/Users/Khaled Khairy/Dropbox/Projects/hot/Project_spherical_parameterization/code\pySHP\tests\cmcf_batch_outputs\1dpx



[2/9] BDH6230_whole_brain.off
Exported spherical harmonics surface to: C:/Users/Khaled Khairy/Dropbox/Projects/hot/Project_spherical_parameterization/code\pySHP\tests\cmcf_batch_outputs\BDH6230_whole_brain\BDH6230_whole_brain_b.shp3
  L_max: 16
  Number of coefficients: 289
  Components: 3 (x, y, z)
Exported spherical harmonics surface to: C:/Users/Khaled Khairy/Dropbox/Projects/hot/Project_spherical_parameterization/code\pySHP\tests\cmcf_batch_outputs\BDH6230_whole_brain\BDH6230_whole_brain_c_invariant.shp3
  L_max: 16
  Number of coefficients: 289
  Components: 3 (x, y, z)
  BDH6230_whole_brain: bijective=True folds=0 RMS=0.68%  -> 5 files in C:/Users/Khaled Khairy/Dropbox/Projects/hot/Project_spherical_parameterization/code\pySHP\tests\cmcf_batch_outputs\BDH6230_whole_brain



[3/9] echinocyte.off
Exported spherical harmonics surface to: C:/Users/Khaled Khairy/Dropbox/Projects/hot/Project_spherical_parameterization/code\pySHP\tests\cmcf_batch_outputs\echinocyte\echinocyte_b.shp3
  L_max: 16
  Number of coefficients: 289
  Components: 3 (x, y, z)
Exported spherical harmonics surface to: C:/Users/Khaled Khairy/Dropbox/Projects/hot/Project_spherical_parameterization/code\pySHP\tests\cmcf_batch_outputs\echinocyte\echinocyte_c_invariant.shp3
  L_max: 16
  Number of coefficients: 289
  Components: 3 (x, y, z)
  echinocyte: bijective=True folds=0 RMS=0.73%  -> 5 files in C:/Users/Khaled Khairy/Dropbox/Projects/hot/Project_spherical_parameterization/code\pySHP\tests\cmcf_batch_outputs\echinocyte



[4/9] hydra_full_smooth.off


## Summary

In [ ]:
cols = ['name', 'n_in', 'genus', 'n_param', 'quality', 'kappa', 'bijective',
        'folds', 'min_q', 'max_shear', 'area_cov', 'shp_rms_pct', 'status']
try:
    import pandas as pd
    df = pd.DataFrame(results)
    for c in cols:
        if c not in df.columns: df[c] = None
    display(df[cols])
except Exception:
    print('  '.join(cols))
    for r in results:
        print('  '.join(str(r.get(c, '')) for c in cols))

ok = sum(1 for r in results if r.get('status') == 'ok')
bij = sum(1 for r in results if r.get('bijective'))
print(f'\n{ok}/{len(results)} succeeded; {bij} bijective (0 foldovers).')
print(f'Artifacts (a-e per mesh) under: {OUT_DIR}')

## Rotation/scale invariance — shape distance demo

The per-degree energy descriptor `cmcf.shp_degree_energy(shp)` is the robust
rotation+scale invariant. Distance between two shapes = Euclidean distance of
their descriptors. (The smaller, the more similar.)

In [ ]:
ok_res = [r for r in results if r.get('status') == 'ok' and r.get('descriptor')]
if len(ok_res) >= 2:
    # Use the in-memory invariant descriptors collected during the batch.
    descs = {r['name']: np.asarray(r['descriptor']) for r in ok_res}
    names = list(descs)
    print('pairwise SHP shape distance (per-degree energy descriptor):')
    print('        ' + '  '.join(f'{n[:8]:>8}' for n in names))
    for a in names:
        row = '  '.join(f'{np.linalg.norm(descs[a]-descs[b]):8.3f}' for b in names)
        print(f'{a[:8]:>8}  {row}')
else:
    print('need >=2 successful meshes for a distance demo')